<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/10-partitioning-shuffle/04-data-skew.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 10.4 — Data skew everywhere

Build a properly skewed dataset, run the three detection probes,
prove that reducible aggregations dodge skew while holistic ones
don't, fix the holistic case with two-phase salting, and time the
window-to-aggregation rewrite.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("10.4-data-skew")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")   # nothing hides the straggler
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## A dataset with a whale in it

3M rows, 1000 keys — but key 0 owns 60% of all rows. This is the
shape of real clickstreams (one bot), ledgers (one corporate
account), and logs (one chatty service).

In [ ]:
skewed = (spark.range(3_000_000)
          .withColumn("key",
                      F.when(F.rand(seed=7) < 0.6, F.lit(0))          # the whale
                       .otherwise((F.rand(seed=8) * 999 + 1).cast("int")))
          .withColumn("v", (F.rand(seed=9) * 1000).cast("double"))
          .withColumn("ts", col("id")))

even = (spark.range(3_000_000)
        .withColumn("key", (col("id") % 1000).cast("int"))
        .withColumn("v", (F.rand(seed=9) * 1000).cast("double"))
        .withColumn("ts", col("id")))

skewed.cache().count(), even.cache().count()

## The three probes

Key histogram, partition histogram, and (after any slow run) the
UI's Summary Metrics — median vs max is the whole diagnosis.

In [ ]:
# probe 2: key histogram — whale within 10× of median? no: ~1800×
skewed.groupBy("key").count().orderBy(F.desc("count")).show(5)

# probe 3: physical layout after hash-partitioning by the key
(skewed.repartition(8, "key")
 .groupBy(F.spark_partition_id().alias("pid")).count()
 .orderBy(F.desc("count")).show(8))
# one monster partition, seven starving — the iron law, photographed

## Reducible vs holistic: the partial-agg discount in action

Same whale, two aggregations. `sum` collapses the whale on the map
side (skewed ≈ even). `collect_list` ships every raw value to one
task (skewed ≫ even). Watch the stage timeline in the UI during
the last run — seven tasks finish instantly, one grinds.

In [ ]:
import time

def bench(label, df):
    start = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:>34}: {time.perf_counter() - start:5.1f} s")

bench("sum, even keys",    even.groupBy("key").agg(F.sum("v")))
bench("sum, whale key",    skewed.groupBy("key").agg(F.sum("v")))
bench("collect_list, even", even.groupBy("key").agg(F.collect_list("v")))
bench("collect_list, whale", skewed.groupBy("key").agg(F.collect_list("v")))

## Fixing the holistic case: two-phase salted aggregation

Phase one spreads the whale over 16 tasks and pre-collects; phase
two merges ≤16 chunks per key. No dimension to inflate — cheaper
than 7.4's join salting.

In [ ]:
SALT = 16

salted = (skewed
    .withColumn("salt", (F.rand(seed=1) * SALT).cast("int"))
    .groupBy("key", "salt")
    .agg(F.collect_list("v").alias("chunk"))          # heavy work, 16-way
    .groupBy("key")
    .agg(F.flatten(F.collect_list("chunk")).alias("vals")))  # merge 16 chunks

bench("collect_list, whale, salted", salted)

# correctness spot-check: same total element count per key
plain = skewed.groupBy("key").agg(F.size(F.collect_list("v")).alias("n"))
twop = salted.select("key", F.size("vals").alias("n"))
print("keys where counts differ:",
      plain.join(twop, "key").where(plain.n != twop.n).count())

## Window skew: the reformulation that deletes it

A per-key max over the unbounded frame forces the whale through one
task when written as a window. As aggregate + broadcast join, the
whale collapses on the map side and never bottlenecks.

In [ ]:
w = Window.partitionBy("key")

as_window = skewed.withColumn("key_max", F.max("v").over(w))

key_max = skewed.groupBy("key").agg(F.max("v").alias("key_max"))
as_agg_join = skewed.join(F.broadcast(key_max), "key")

bench("window over whale key", as_window)
bench("agg + broadcast join", as_agg_join)

# same answer, different physics
print("rows that differ:",
      as_window.select("id", "key_max")
               .exceptAll(as_agg_join.select("id", "key_max")).count())

In [ ]:
skewed.unpersist(), even.unpersist()   # self-clean (cache only, no files this time)

## Exercises

1. Add a null whale: make 20% of `skewed`'s keys null and rerun
   probe 2 and the `sum` benchmarks. Does null behave differently
   from key 0 in an aggregation? Should it? (Compare 7.4's join
   case, where null-splitting was a real fix.)
2. Find the salt sweet spot: rerun the two-phase fix with SALT in
   {2, 8, 16, 64, 256}. Plot (or tabulate) the times and explain
   both ends of the curve.
3. The two-phase trick also works for exact `countDistinct` —
   phase one `collect_set` per (key, salt)... or is there something
   better? Write it with `array_distinct`/`flatten`, then compare
   against `approx_count_distinct` on time and error.
4. Break the window rewrite: change the requirement to "rank of
   each row within its key by ts". Show why agg+join can't express
   it, then implement the hot/cold split (keys > 100k rows via a
   different path) and measure.
5. Turn probe 1 into code: using 10.2's REST-API helper, fetch
   `/stages/{id}/{attempt}/taskSummary` for the collect_list-whale
   stage and print the 0.5 and 1.0 duration quantiles. Wrap it as
   `skew_score(stage_id)` — you'll want it in Module 13.

In [ ]:
# your exercise code here